# AMEX Enterprise Credit Risk Platform
## Notebook 13 — Power BI Dashboard: Star-Schema Data Model, DAX Measures & Import-Ready Exports
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Deployment / Business Intelligence**. Notebook 13 of 18. Depends on Notebooks 01, 04 and 05 (reuses Notebook 05's real saved champion model and fitted preprocessing artifacts, as-is); Notebook 09's model registry and Notebook 12's monitoring windows are used opportunistically if present.

**An honest limitation, stated up front.** Power BI Desktop is a proprietary Windows GUI application. There is no Python library that produces a genuine, openable `.pbix` file, and this platform's standing rule is to never claim unsupported capability — so this notebook does not attempt to fabricate one. What it does instead, all for real: build a proper star-schema data model (one fact table, real dimension tables) from this platform's real, live-computed scoring output; export it as both CSV and Parquet (Power BI opens either natively); write real, syntactically valid DAX measure definitions ready to paste into Power BI Desktop; generate a real star-schema relationship diagram with live-computed cardinalities; and render a static HTML **preview** that approximates the intended report layout — clearly labeled as a preview, not a substitute for the real interactive `.pbix`, which must be built in Power BI Desktop from the files this notebook produces.

**Risk-tier bands are a stated business-policy ASSUMPTION.** The Kaggle dataset does not define "Prime" / "Subprime" cut points — this notebook picks a common, editable set of PD thresholds and labels them exactly that.

**Deliverables:** `fact_customer_risk_scores.csv` + `.parquet`, `dim_risk_tier.csv`, `dim_model_version.csv`, `dim_monitoring_window.csv` (if Notebook 12 has run), `dax_measures.dax`, `star_schema_diagram.png`, `PowerBI_Dashboard_Preview.html`, `powerbi_import_guide.docx` section, `powerbi_readiness_checklist.csv`, and `PowerBI_Dashboard_Report.docx`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01, 04, 05
# =============================================================================
import os
import sys
import csv
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01, 04, 05")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB04_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_04_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB09_MLOPS_REGISTRY_PATH = None  # resolved after PILLAR_DIRS is known, below
NB12_WINDOWS_PATH = None         # resolved after PILLAR_DIRS is known, below

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB04_SUMMARY_PATH, "run 04_feature_engineering.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix} -- this notebook reads its outputs.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB04_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB04_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)

MODEL_DEV_DIR = PILLAR_DIRS["model_development"]
MODELS_SUBDIR = MODEL_DEV_DIR / "models"
MLOPS_DIR = PILLAR_DIRS["mlops"]
MONITORING_DIR = PILLAR_DIRS["monitoring"]
POWERBI_DIR = PILLAR_DIRS["powerbi_dashboard"]
POWERBI_DIR.mkdir(parents=True, exist_ok=True)

NB09_MLOPS_REGISTRY_PATH = MLOPS_DIR / "model_registry.json"          # optional
NB12_WINDOWS_PATH = MONITORING_DIR / "monitoring_windows_report.csv"  # optional

TRAIN_SPLIT_ENG_PATH = Path(NB04_SUMMARY["output_files"]["train_split_engineered.csv"])
TEST_SPLIT_ENG_PATH = Path(NB04_SUMMARY["output_files"]["test_split_engineered.csv"])
MODEL_COMPARISON_PATH = MODEL_DEV_DIR / "model_comparison.csv"
CHAMPION_IMPORTANCE_PATH = MODEL_DEV_DIR / "champion_feature_importance.csv"
PREPROCESSING_PATH = MODELS_SUBDIR / "preprocessing_artifacts.joblib"

# --- Champion identification: same resilient pattern as Notebooks 06/07/08/09/10/12 --
NB05_SUMMARY = None
if NB05_SUMMARY_PATH.exists():
    with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
        NB05_SUMMARY = json.load(f)
    CHAMPION_NAME = NB05_SUMMARY["champion_model"]
    _champion_source = f"{NB05_SUMMARY_PATH.name}"
elif MODEL_COMPARISON_PATH.exists():
    with open(MODEL_COMPARISON_PATH, "r", encoding="utf-8", newline="") as _f:
        _cmp_rows = list(csv.DictReader(_f))
    if not _cmp_rows or "model" not in _cmp_rows[0] or "holdout_amex_metric" not in _cmp_rows[0]:
        raise RuntimeError(f"{MODEL_COMPARISON_PATH} exists but is missing the expected 'model' / "
                            f"'holdout_amex_metric' columns -- cannot identify a champion from it. "
                            f"Fix: re-run 05_model_development.ipynb.")
    _champion_row = max(_cmp_rows, key=lambda r: float(r["holdout_amex_metric"]))
    CHAMPION_NAME = _champion_row["model"]
    _champion_source = f"{MODEL_COMPARISON_PATH.name} (fallback -- {NB05_SUMMARY_PATH.name} not found)"
else:
    raise FileNotFoundError(f"Neither {NB05_SUMMARY_PATH} nor {MODEL_COMPARISON_PATH} was found.\n"
                             f"Fix: run 05_model_development.ipynb first -- this notebook reuses its saved "
                             f"champion model and preprocessing artifacts.")

CHAMPION_MODEL_PATH = MODELS_SUBDIR / f"{CHAMPION_NAME}.joblib"

for _p in (TRAIN_SPLIT_ENG_PATH, TEST_SPLIT_ENG_PATH, CHAMPION_MODEL_PATH, PREPROCESSING_PATH,
           MODEL_COMPARISON_PATH, CHAMPION_IMPORTANCE_PATH):
    if not _p.exists():
        raise FileNotFoundError(f"Required file not found: {_p}\nFix: re-run 05_model_development.ipynb -- "
                                 f"this notebook reuses its saved champion model and outputs as-is.")

NB09_REGISTRY = None
if NB09_MLOPS_REGISTRY_PATH.exists():
    with open(NB09_MLOPS_REGISTRY_PATH, "r", encoding="utf-8") as f:
        NB09_REGISTRY = json.load(f)

NB12_WINDOWS_DF = None
import pandas as pd
if NB12_WINDOWS_PATH.exists():
    NB12_WINDOWS_DF = pd.read_csv(NB12_WINDOWS_PATH)

print(f"Champion model              : {CHAMPION_NAME}  (identified from: {_champion_source})")
_nb09_status = f"found -- {len(NB09_REGISTRY['entries'])} entries" if NB09_REGISTRY else "not found -- skipping (not required)"
print(f"Notebook 09 model registry   : {_nb09_status}")
print(f"Notebook 12 monitoring windows: {'found -- will link' if NB12_WINDOWS_DF is not None else 'not found -- skipping (not required)'}")
print(f"Power BI artifacts will be written under: {POWERBI_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import gc

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from docx import Document
    from docx.shared import Inches
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4, Concurrency)")


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
_live_vm = psutil.virtual_memory()
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(_live_vm.available * ADAPTIVE_RAM_FRACTION)

print(f"Adaptive RAM ceiling (this run) : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: LOAD CHAMPION MODEL, PREPROCESSING ARTIFACTS & FEATURE IMPORTANCE
# =============================================================================
_section("SECTION 3: Load Champion Model, Preprocessing Artifacts & Feature Importance")

_t0 = time.time()
champion_model = joblib.load(CHAMPION_MODEL_PATH)
preprocessing_artifacts = joblib.load(PREPROCESSING_PATH)
print(f"Loaded champion model '{CHAMPION_NAME}' from {CHAMPION_MODEL_PATH} ({time.time() - _t0:.1f}s)")

label_encoders = preprocessing_artifacts["label_encoders"]
feature_medians = preprocessing_artifacts["feature_medians"]
scaler = preprocessing_artifacts["scaler"]
all_feature_cols = preprocessing_artifacts["all_feature_cols"]
categorical_encode_cols = preprocessing_artifacts["categorical_encode_cols"]
numeric_feature_cols = preprocessing_artifacts["numeric_feature_cols"]
champion_uses_scaled = CHAMPION_NAME == "logistic_regression"

champion_importance_df = pd.read_csv(CHAMPION_IMPORTANCE_PATH)
TOP_K_FEATURES = min(8, len(champion_importance_df))
top_feature_list = champion_importance_df.head(TOP_K_FEATURES)["feature"].tolist()

print(f"Feature columns loaded  : {len(all_feature_cols)} "
      f"({len(numeric_feature_cols)} numeric + {len(categorical_encode_cols)} categorical)")
print(f"Top-{TOP_K_FEATURES} features (by champion importance) carried into the fact table: {top_feature_list}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LOAD HOLDOUT ENGINEERED DATA, APPLY SAVED PREPROCESSING, SCORE
# =============================================================================
_section("SECTION 4: Load Holdout Engineered Data, Apply Saved Preprocessing, Score")

SPLIT_CSV_SCHEMA = {"customer_ID": pl.Utf8, "target": pl.Int8}
for _c in categorical_encode_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Utf8
for _c in numeric_feature_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Float32

_t0 = time.time()
train_pl = pl.read_csv(str(TRAIN_SPLIT_ENG_PATH), schema_overrides=SPLIT_CSV_SCHEMA)
holdout_pl = pl.read_csv(str(TEST_SPLIT_ENG_PATH), schema_overrides=SPLIT_CSV_SCHEMA)
print(f"Loaded test_split_engineered.csv : {holdout_pl.shape[0]:,} x {holdout_pl.shape[1]} (held-out, never trained on)")

_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite() | pl.col(c).is_nan()).then(None).otherwise(pl.col(c)).cast(pl.Float32).alias(c)
    for c in numeric_feature_cols
]
train_pl = train_pl.with_columns(_inf_clean_exprs)
holdout_pl = holdout_pl.with_columns(_inf_clean_exprs)

for c in categorical_encode_cols:
    train_pl = train_pl.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
    holdout_pl = holdout_pl.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
    _mapping = {cat: i for i, cat in enumerate(label_encoders[c]["classes"])}
    train_pl = train_pl.with_columns(pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))
    holdout_pl = holdout_pl.with_columns(pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))

_impute_exprs = [pl.col(c).fill_null(feature_medians[c]) for c in numeric_feature_cols]
train_pl = train_pl.with_columns(_impute_exprs)
holdout_pl = holdout_pl.with_columns(_impute_exprs)

X_train = train_pl.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
_train_mean = scaler["mean"]
_train_std = scaler["std"]

X_holdout = holdout_pl.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
y_holdout = holdout_pl.get_column("target").to_numpy().astype(np.int64, copy=False)
holdout_customer_ids = holdout_pl.get_column("customer_ID").to_numpy()
X_holdout_scaled = (X_holdout - _train_mean) / _train_std

_top_feature_pos = {f: all_feature_cols.index(f) for f in top_feature_list}
_top_feature_values = {f: X_holdout[:, _top_feature_pos[f]] for f in top_feature_list}

Xc_holdout = X_holdout_scaled if champion_uses_scaled else X_holdout
PD_HOLDOUT = champion_model.predict_proba(Xc_holdout)[:, 1]

del train_pl, holdout_pl
gc.collect()
print(f"Scored {X_holdout.shape[0]:,} holdout customers with the real champion model ({time.time() - _t0:.1f}s total)")
print(f"Process RSS now: {_rss_gb():.2f} GB (of {MAX_RAM_BYTES / 1e9:.2f} GB adaptive ceiling)")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: RISK-TIER BANDS (STATED ASSUMPTION) & FACT/DIMENSION TABLES
# =============================================================================
_section("SECTION 5: Risk-Tier Bands (Stated Assumption) & Fact/Dimension Tables")

# --- ASSUMPTION: the Kaggle dataset does not define "Prime" / "Subprime" cut
#     points -- there is no ground truth to compute these from. This is a
#     common, editable industry-style convention, not a fact this notebook
#     measured. Change PD_TIER_BANDS to match your organization's real credit
#     policy before using this in production. ---
PD_TIER_BANDS = [
    {"risk_tier": "Prime", "tier_order": 1, "pd_lower": 0.00, "pd_upper": 0.05,
     "description": "ASSUMPTION -- lowest-risk band; typically auto-approved"},
    {"risk_tier": "Near-Prime", "tier_order": 2, "pd_lower": 0.05, "pd_upper": 0.15,
     "description": "ASSUMPTION -- approved with standard terms"},
    {"risk_tier": "Subprime", "tier_order": 3, "pd_lower": 0.15, "pd_upper": 0.35,
     "description": "ASSUMPTION -- approved with risk-adjusted pricing / limits, or manual review"},
    {"risk_tier": "High Risk", "tier_order": 4, "pd_lower": 0.35, "pd_upper": 1.01,
     "description": "ASSUMPTION -- typically declined or requires senior underwriter override"},
]


def _assign_tier(pd_value: float) -> str:
    for band in PD_TIER_BANDS:
        if band["pd_lower"] <= pd_value < band["pd_upper"]:
            return band["risk_tier"]
    return PD_TIER_BANDS[-1]["risk_tier"]


risk_tier_assignments = np.array([_assign_tier(p) for p in PD_HOLDOUT])

# --- Optional link to Notebook 12's simulated monitoring windows: reuse the
#     EXACT same row-order partition logic Notebook 12 uses, so window_id is
#     consistent whether or not notebook_12's own output file is present.
#     This is a real, live computation either way -- never fabricated. ---
N_WINDOWS_REQUESTED = 6
MIN_WINDOW_SIZE = 300
_holdout_n = X_holdout.shape[0]
N_MONITORING_WINDOWS = max(1, min(N_WINDOWS_REQUESTED, _holdout_n // MIN_WINDOW_SIZE)) if _holdout_n >= MIN_WINDOW_SIZE else 1
_window_index_splits = np.array_split(np.arange(_holdout_n), N_MONITORING_WINDOWS)
window_id_by_row = np.zeros(_holdout_n, dtype=np.int64)
for _widx, _ridx in enumerate(_window_index_splits):
    window_id_by_row[_ridx] = _widx + 1

# --- FACT TABLE: one row per holdout customer, real predicted PD, real actual
#     outcome, the ASSUMPTION-banded risk tier, top-K real feature values, and
#     the champion model's registry version if Notebook 09 has run. ---
_fact_data = {
    "customer_ID": holdout_customer_ids,
    "predicted_pd": np.round(PD_HOLDOUT, 6),
    "actual_default": y_holdout,
    "risk_tier": risk_tier_assignments,
    "monitoring_window_id": window_id_by_row,
}
for f in top_feature_list:
    _fact_data[f"feat_{f}"] = np.round(_top_feature_values[f], 6)

_champion_version = None
if NB09_REGISTRY:
    _matches = [e for e in NB09_REGISTRY["entries"] if e.get("model_name") == CHAMPION_NAME]
    if _matches:
        _champion_version = max(_matches, key=lambda e: e["version"])["version"]
_fact_data["model_version"] = _champion_version if _champion_version is not None else "unversioned (run Notebook 09)"

fact_df = pd.DataFrame(_fact_data)
print(f"fact_customer_risk_scores: {fact_df.shape[0]:,} rows x {fact_df.shape[1]} columns")
print(fact_df["risk_tier"].value_counts().to_string())
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: DIMENSION TABLES
# =============================================================================
_section("SECTION 6: Dimension Tables")

dim_risk_tier_df = pd.DataFrame(PD_TIER_BANDS)

if NB09_REGISTRY and NB09_REGISTRY["entries"]:
    dim_model_version_df = pd.DataFrame(NB09_REGISTRY["entries"])
    _model_version_source = f"{NB09_MLOPS_REGISTRY_PATH.name} ({len(dim_model_version_df)} version(s))"
else:
    dim_model_version_df = pd.DataFrame([{
        "model_name": CHAMPION_NAME, "version": "unversioned", "registered_at_utc": None, "sha256": None,
        "file_size_mb": None, "holdout_auc": None, "holdout_amex_metric": None, "holdout_top4pct_capture": None,
        "risk_tier": None, "latency_p50_ms": None, "latency_p99_ms": None, "batch_throughput_rows_per_sec": None,
    }])
    _model_version_source = "fallback single row -- Notebook 09 has not been run yet"

if NB12_WINDOWS_DF is not None:
    dim_monitoring_window_df = NB12_WINDOWS_DF.copy()
    _monitoring_window_source = f"{NB12_WINDOWS_PATH.name} ({len(dim_monitoring_window_df)} window(s), Notebook 12's own real computation)"
else:
    dim_monitoring_window_df = pd.DataFrame({
        "window": list(range(1, N_MONITORING_WINDOWS + 1)),
        "n_customers": [len(s) for s in _window_index_splits],
    })
    _monitoring_window_source = "self-computed here (Notebook 12 has not been run yet) -- window boundaries only, no drift metrics"

print(f"dim_risk_tier            : {len(dim_risk_tier_df)} rows")
print(f"dim_model_version        : {_model_version_source}")
print(f"dim_monitoring_window    : {_monitoring_window_source}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: EXPORT FACT & DIMENSION TABLES (CSV + PARQUET, POWER BI READY)
# =============================================================================
_section("SECTION 7: Export Fact & Dimension Tables (CSV + Parquet, Power BI Ready)")

fact_csv_path = POWERBI_DIR / "fact_customer_risk_scores.csv"
fact_parquet_path = POWERBI_DIR / "fact_customer_risk_scores.parquet"
dim_risk_tier_path = POWERBI_DIR / "dim_risk_tier.csv"
dim_model_version_path = POWERBI_DIR / "dim_model_version.csv"
dim_monitoring_window_path = POWERBI_DIR / "dim_monitoring_window.csv"

fact_df.to_csv(fact_csv_path, index=False)
pl.from_pandas(fact_df).write_parquet(fact_parquet_path)  # Polars write -- real columnar export for Direct Query/large scale
dim_risk_tier_df.to_csv(dim_risk_tier_path, index=False)
dim_model_version_df.to_csv(dim_model_version_path, index=False)
dim_monitoring_window_df.to_csv(dim_monitoring_window_path, index=False)

print(f"\u2705 Saved -> {fact_csv_path}  ({fact_csv_path.stat().st_size / 1e6:.2f} MB)")
print(f"\u2705 Saved -> {fact_parquet_path}  ({fact_parquet_path.stat().st_size / 1e6:.2f} MB)")
print(f"\u2705 Saved -> {dim_risk_tier_path}")
print(f"\u2705 Saved -> {dim_model_version_path}")
print(f"\u2705 Saved -> {dim_monitoring_window_path}")
print("\nIn Power BI Desktop: Get Data -> Text/CSV (or Parquet) -> select each file above -> Model view -> "
      "draw the relationships below (Section 8 documents them, with live-computed cardinalities).")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: STAR SCHEMA -- RELATIONSHIPS, CARDINALITIES & DIAGRAM
# =============================================================================
_section("SECTION 8: Star Schema -- Relationships, Cardinalities & Diagram")

star_schema_relationships = [
    {"from_table": "dim_risk_tier", "from_column": "risk_tier", "to_table": "fact_customer_risk_scores",
     "to_column": "risk_tier", "cardinality": f"1:{len(fact_df):,}",
     "cross_filter": "Single (dim -> fact)"},
    {"from_table": "dim_model_version", "from_column": "version", "to_table": "fact_customer_risk_scores",
     "to_column": "model_version", "cardinality": f"1:{len(fact_df):,}",
     "cross_filter": "Single (dim -> fact)"},
    {"from_table": "dim_monitoring_window", "from_column": "window", "to_table": "fact_customer_risk_scores",
     "to_column": "monitoring_window_id", "cardinality": f"1:{len(fact_df):,}",
     "cross_filter": "Single (dim -> fact)"},
]
star_schema_df = pd.DataFrame(star_schema_relationships)
print(star_schema_df.to_string(index=False))

PROBLEM_NAME = "Phase 1 \u00b7 Problem 1 -- Credit Scoring / PD Prediction"
VIZ = {"surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
       "cat_blue": "#2a78d6", "cat_red": "#e34948", "cat_green": "#3a9e5f", "cat_amber": "#d69a2a",
       "fact_fill": "#2a78d6", "dim_fill": "#3a9e5f"}

fig, ax = plt.subplots(figsize=(9, 6), dpi=150)
ax.set_facecolor(VIZ["surface"]); fig.set_facecolor(VIZ["surface"])
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")

_fact_box = dict(boxstyle="round,pad=0.5", facecolor=VIZ["fact_fill"], edgecolor="none", alpha=0.92)
_dim_box = dict(boxstyle="round,pad=0.5", facecolor=VIZ["dim_fill"], edgecolor="none", alpha=0.92)

ax.text(5, 5, f"fact_customer_risk_scores\n({len(fact_df):,} rows)", ha="center", va="center",
        fontsize=10, color="white", weight="bold", bbox=_fact_box)
_dim_positions = [(1.6, 8.5, "dim_risk_tier"), (8.4, 8.5, "dim_model_version"), (5, 1.5, "dim_monitoring_window")]
for _x, _y, _label in _dim_positions:
    ax.text(_x, _y, f"{_label}\n({len(dim_risk_tier_df) if _label=='dim_risk_tier' else (len(dim_model_version_df) if _label=='dim_model_version' else len(dim_monitoring_window_df))} rows)",
            ha="center", va="center", fontsize=9, color="white", weight="bold", bbox=_dim_box)
    ax.annotate("", xy=(5, 5.5) if _y > 5 else (5, 4.5), xytext=(_x, _y - 0.6 if _y > 5 else _y + 0.6),
                arrowprops=dict(arrowstyle="->", color=VIZ["text_secondary"], lw=1.4))

ax.set_title(f"{PROBLEM_NAME}\nPower BI Star Schema -- Live-Computed Cardinalities", fontsize=11, color=VIZ["text_primary"])
fig.tight_layout()
star_schema_chart_path = POWERBI_DIR / "star_schema_diagram.png"
fig.savefig(star_schema_chart_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {star_schema_chart_path}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: DAX MEASURES (REAL, VALID SYNTAX -- READY TO PASTE INTO POWER BI)
# =============================================================================
_section("SECTION 9: DAX Measures (Real, Valid Syntax -- Ready to Paste Into Power BI)")

# --- Real, syntactically valid DAX. Honest scope note: there is no Python DAX
#     engine to execute these against the model live -- exactly like Notebook
#     09's CI/CD YAML, this is a real, ready-to-use artifact you paste into
#     Power BI Desktop's Model view (New Measure), not something this notebook
#     can run itself. Every column referenced below is a real column this
#     notebook actually exported in Section 7 -- nothing here references a
#     field that does not exist in the model. ---
DAX_MEASURE_DEFINITIONS = [
    "Total Accounts = COUNTROWS(fact_customer_risk_scores)",
    "Total Actual Defaults = SUM(fact_customer_risk_scores[actual_default])",
    "Actual Default Rate = "
    "DIVIDE([Total Actual Defaults], [Total Accounts], 0)",
    "Average Predicted PD = "
    "AVERAGE(fact_customer_risk_scores[predicted_pd])",
    "High Risk Account Count = "
    "CALCULATE([Total Accounts], fact_customer_risk_scores[risk_tier] = \"High Risk\")",
    "High Risk Share = "
    "DIVIDE([High Risk Account Count], [Total Accounts], 0)",
    "Prime Share = "
    "DIVIDE(CALCULATE([Total Accounts], fact_customer_risk_scores[risk_tier] = \"Prime\"), [Total Accounts], 0)",
    "Rank-Ordering Check (Avg PD by Tier) = "
    "AVERAGEX(VALUES(dim_risk_tier[risk_tier]), CALCULATE([Average Predicted PD]))",
    "Model Calibration Gap = "
    "[Average Predicted PD] - [Actual Default Rate]",
]
N_DAX_MEASURES = len(DAX_MEASURE_DEFINITIONS)
DAX_MEASURES = "\n\n".join(DAX_MEASURE_DEFINITIONS)

dax_measures_path = POWERBI_DIR / "dax_measures.dax"
with open(dax_measures_path, "w", encoding="utf-8") as f:
    f.write(DAX_MEASURES)
print(DAX_MEASURES)
print(f"\n\u2705 Saved -> {dax_measures_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: CHARTS (STATIC, WHAT THE POWER BI REPORT SHOULD REPLICATE)
# =============================================================================
_section("SECTION 10: Charts (Static -- What the Power BI Report Should Replicate)")


def _style_axes(ax):
    ax.set_facecolor(VIZ["surface"]); ax.figure.set_facecolor(VIZ["surface"])
    ax.grid(axis="y", color=VIZ["grid"], linewidth=0.8, zorder=0); ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(VIZ["grid"])
    ax.tick_params(colors=VIZ["text_secondary"], labelsize=9)
    ax.title.set_color(VIZ["text_primary"])
    ax.xaxis.label.set_color(VIZ["text_secondary"]); ax.yaxis.label.set_color(VIZ["text_secondary"])


_tier_order = [b["risk_tier"] for b in PD_TIER_BANDS]
_tier_counts = fact_df["risk_tier"].value_counts().reindex(_tier_order).fillna(0)
_tier_actual_rate = fact_df.groupby("risk_tier")["actual_default"].mean().reindex(_tier_order)

# Chart 1: account distribution by risk tier
fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=150)
_bars = ax.bar(_tier_order, _tier_counts.values, color=VIZ["cat_blue"], zorder=3)
ax.bar_label(_bars, padding=3, fontsize=9, color=VIZ["text_primary"], fmt="%.0f")
_style_axes(ax)
ax.set_ylabel("Number of accounts (holdout, real)")
ax.set_title(f"{PROBLEM_NAME}\nAccount Distribution by Risk Tier (ASSUMPTION bands)", fontsize=11)
fig.tight_layout()
chart1_path = POWERBI_DIR / "risk_tier_distribution_chart.png"
fig.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart1_path}")

# Chart 2: actual default rate by risk tier (does the tiering actually rank-order risk?)
fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=150)
_bars = ax.bar(_tier_order, (_tier_actual_rate * 100).values, color=VIZ["cat_red"], zorder=3)
ax.bar_label(_bars, padding=3, fontsize=9, color=VIZ["text_primary"], fmt="%.1f%%")
_style_axes(ax)
ax.set_ylabel("Actual default rate (%, holdout, real)")
ax.set_title(f"{PROBLEM_NAME}\nActual Default Rate by Risk Tier -- Validates the Tiering", fontsize=11)
fig.tight_layout()
chart2_path = POWERBI_DIR / "default_rate_by_tier_chart.png"
fig.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart2_path}")

# Chart 3: predicted PD distribution
fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=150)
ax.hist(fact_df["predicted_pd"], bins=40, color=VIZ["cat_blue"], zorder=3)
_style_axes(ax)
ax.set_xlabel("Predicted PD"); ax.set_ylabel("Number of accounts")
ax.set_title(f"{PROBLEM_NAME}\nPredicted PD Distribution -- Holdout Population (Real)", fontsize=11)
fig.tight_layout()
chart3_path = POWERBI_DIR / "pd_distribution_chart.png"
fig.savefig(chart3_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart3_path}")

print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: HTML PREVIEW -- APPROXIMATES THE INTENDED POWER BI REPORT LAYOUT
# =============================================================================
_section("SECTION 11: HTML Preview -- Approximates the Intended Power BI Report Layout")

# --- Built via a plain (non-f) triple-double-quoted template with a single
#     token substituted via .replace() -- this platform's established
#     convention for avoiding nested-quote collisions with this cell's own
#     r\'\'\'...\'\'\' wrapper (see Notebook 14's dashboard generation). This is a
#     STATIC preview -- clearly labeled as such in its own banner -- not a
#     substitute for the real, interactive .pbix. ---
_preview_data = {
    "risk_tiers": _tier_order,
    "tier_counts": [int(v) for v in _tier_counts.values],
    "tier_actual_rate_pct": [round(float(v) * 100, 2) for v in _tier_actual_rate.values],
    "pd_histogram": {
        "counts": [int(c) for c in np.histogram(fact_df["predicted_pd"], bins=20)[0]],
        "bin_edges": [round(float(e), 4) for e in np.histogram(fact_df["predicted_pd"], bins=20)[1]],
    },
    "total_accounts": int(len(fact_df)),
    "actual_default_rate_pct": round(float(fact_df["actual_default"].mean()) * 100, 2),
    "avg_predicted_pd_pct": round(float(fact_df["predicted_pd"].mean()) * 100, 2),
    "model_version": str(_champion_version) if _champion_version is not None else "unversioned",
    "champion_model": CHAMPION_NAME,
    "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
}
_preview_json = json.dumps(_preview_data)

HTML_TEMPLATE = """<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>AMEX PD Scoring -- Power BI Report Preview (Static)</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.0/chart.umd.min.js"></script>
<style>
  body { font-family: -apple-system, Segoe UI, Arial, sans-serif; margin: 0; background: #f4f4f2; color: #0b0b0b; }
  header { background: #0b2a4a; color: white; padding: 20px 28px; }
  header h1 { margin: 0 0 4px 0; font-size: 20px; }
  header p { margin: 0; font-size: 13px; opacity: 0.85; }
  .banner { background: #d69a2a; color: #2a1c00; padding: 10px 28px; font-size: 13px; font-weight: 600; }
  .kpis { display: flex; gap: 16px; padding: 20px 28px; flex-wrap: wrap; }
  .kpi { background: white; border-radius: 8px; padding: 16px 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.08); min-width: 170px; }
  .kpi .label { font-size: 12px; color: #52514e; }
  .kpi .value { font-size: 24px; font-weight: 700; color: #0b2a4a; }
  .charts { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; padding: 0 28px 28px 28px; }
  .chart-card { background: white; border-radius: 8px; padding: 16px; box-shadow: 0 1px 3px rgba(0,0,0,0.08); }
  .chart-card h3 { margin: 0 0 10px 0; font-size: 14px; color: #0b2a4a; }
  canvas { max-height: 280px; }
  .fallback { display: none; padding: 40px; text-align: center; color: #52514e; }
  footer { padding: 16px 28px; font-size: 12px; color: #52514e; }
</style>
</head>
<body>
<header>
  <h1>AMEX Enterprise Credit Risk Platform -- Power BI Report Preview</h1>
  <p>Phase 1 &middot; Problem 1 -- Credit Scoring / PD Prediction &middot; Champion: __CHAMPION_MODEL__ (model version __MODEL_VERSION__) &middot; Generated __GENERATED_AT__</p>
</header>
<div class="banner">STATIC PREVIEW ONLY -- this approximates the intended Power BI report layout. It is not the real, interactive .pbix. Build that in Power BI Desktop using fact_customer_risk_scores.csv/.parquet, the dim_*.csv files, and dax_measures.dax in this same folder.</div>
<div class="kpis">
  <div class="kpi"><div class="label">Total Accounts (Holdout)</div><div class="value" id="kpiTotal">--</div></div>
  <div class="kpi"><div class="label">Actual Default Rate</div><div class="value" id="kpiDefaultRate">--</div></div>
  <div class="kpi"><div class="label">Average Predicted PD</div><div class="value" id="kpiAvgPd">--</div></div>
</div>
<div class="charts">
  <div class="chart-card"><h3>Account Distribution by Risk Tier</h3><canvas id="tierCountChart"></canvas></div>
  <div class="chart-card"><h3>Actual Default Rate by Risk Tier</h3><canvas id="tierRateChart"></canvas></div>
  <div class="chart-card"><h3>Predicted PD Distribution</h3><canvas id="pdHistChart"></canvas></div>
  <div class="chart-card"><h3>Model Calibration (Avg Predicted PD vs. Actual Default Rate)</h3><canvas id="calibrationChart"></canvas></div>
</div>
<div class="fallback" id="fallbackBanner">Chart.js could not load from the CDN in this environment -- the KPI cards above are still real, live-computed numbers. Open this file with internet access to see the charts, or open the real .pbix in Power BI Desktop instead.</div>
<footer>Deliberately static: no drill-down, no cross-filtering. Power BI Desktop provides that; this HTML file does not attempt to replicate it.</footer>
<script>
const DATA = __DASHBOARD_DATA_JSON__;
document.getElementById("kpiTotal").textContent = DATA.total_accounts.toLocaleString();
document.getElementById("kpiDefaultRate").textContent = DATA.actual_default_rate_pct.toFixed(2) + "%";
document.getElementById("kpiAvgPd").textContent = DATA.avg_predicted_pd_pct.toFixed(2) + "%";

const CHART_LIB_AVAILABLE = (typeof Chart !== "undefined");

function safeRenderChart(fn, label) {
  if (!CHART_LIB_AVAILABLE) {
    document.getElementById("fallbackBanner").style.display = "block";
    return;
  }
  try { fn(); } catch (e) { console.error("Chart render failed: " + label, e); }
}

function renderTierCountChart() {
  new Chart(document.getElementById("tierCountChart"), {
    type: "bar",
    data: { labels: DATA.risk_tiers, datasets: [{ label: "Accounts", data: DATA.tier_counts, backgroundColor: "#2a78d6" }] },
    options: { plugins: { legend: { display: false } }, scales: { y: { beginAtZero: true } } }
  });
}
function renderTierRateChart() {
  new Chart(document.getElementById("tierRateChart"), {
    type: "bar",
    data: { labels: DATA.risk_tiers, datasets: [{ label: "Actual Default Rate (%)", data: DATA.tier_actual_rate_pct, backgroundColor: "#e34948" }] },
    options: { plugins: { legend: { display: false } }, scales: { y: { beginAtZero: true } } }
  });
}
function renderPdHistChart() {
  const edges = DATA.pd_histogram.bin_edges;
  const labels = edges.slice(0, -1).map((e, i) => e.toFixed(2) + "-" + edges[i+1].toFixed(2));
  new Chart(document.getElementById("pdHistChart"), {
    type: "bar",
    data: { labels: labels, datasets: [{ label: "Accounts", data: DATA.pd_histogram.counts, backgroundColor: "#3a9e5f" }] },
    options: { plugins: { legend: { display: false } }, scales: { x: { ticks: { maxRotation: 90, minRotation: 90 } } } }
  });
}
function renderCalibrationChart() {
  new Chart(document.getElementById("calibrationChart"), {
    type: "bar",
    data: { labels: ["Average Predicted PD", "Actual Default Rate"],
             datasets: [{ label: "%", data: [DATA.avg_predicted_pd_pct, DATA.actual_default_rate_pct], backgroundColor: ["#2a78d6", "#e34948"] }] },
    options: { plugins: { legend: { display: false } }, scales: { y: { beginAtZero: true } } }
  });
}

safeRenderChart(renderTierCountChart, "tier count");
safeRenderChart(renderTierRateChart, "tier rate");
safeRenderChart(renderPdHistChart, "pd histogram");
safeRenderChart(renderCalibrationChart, "calibration");
</script>
</body>
</html>
"""

_html_out = (HTML_TEMPLATE
             .replace("__DASHBOARD_DATA_JSON__", _preview_json)
             .replace("__CHAMPION_MODEL__", CHAMPION_NAME)
             .replace("__MODEL_VERSION__", str(_champion_version) if _champion_version is not None else "unversioned")
             .replace("__GENERATED_AT__", datetime.now().strftime("%Y-%m-%d %H:%M")))

html_preview_path = POWERBI_DIR / "PowerBI_Dashboard_Preview.html"
with open(html_preview_path, "w", encoding="utf-8") as f:
    f.write(_html_out)
print(f"\u2705 Saved -> {html_preview_path}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: POWER BI READINESS CHECKLIST
# =============================================================================
_section("SECTION 12: Power BI Readiness Checklist")

powerbi_checklist = [
    {"dimension": "Fact Table Exported (CSV + Parquet)", "status": "Pass", "evidence": f"{fact_csv_path.name}, {fact_parquet_path.name}"},
    {"dimension": "Dimension Tables Exported", "status": "Pass", "evidence": "dim_risk_tier, dim_model_version, dim_monitoring_window"},
    {"dimension": "Star Schema Relationships Documented", "status": "Pass", "evidence": f"{len(star_schema_df)} relationship(s), live cardinalities"},
    {"dimension": "DAX Measures Written (Real, Valid Syntax)", "status": "Pass", "evidence": f"{dax_measures_path.name}, {N_DAX_MEASURES} measures"},
    {"dimension": "Real .pbix Built in Power BI Desktop", "status": "Not Yet Completed",
     "evidence": "Requires Power BI Desktop (Windows GUI app) -- this notebook cannot produce one, by design (see intro)"},
    {"dimension": "HTML Static Preview Generated", "status": "Pass", "evidence": html_preview_path.name},
    {"dimension": "Risk-Tier Bands Documented as Assumption", "status": "Pass", "evidence": "PD_TIER_BANDS, editable"},
    {"dimension": "Model Version Dimension Linked to Notebook 09", "status": "Pass" if NB09_REGISTRY else "Fallback (Notebook 09 not yet run)",
     "evidence": _model_version_source},
    {"dimension": "Monitoring Window Dimension Linked to Notebook 12", "status": "Pass" if NB12_WINDOWS_DF is not None else "Fallback (Notebook 12 not yet run)",
     "evidence": _monitoring_window_source},
]
powerbi_checklist_df = pd.DataFrame(powerbi_checklist)
powerbi_checklist_path = POWERBI_DIR / "powerbi_readiness_checklist.csv"
powerbi_checklist_df.to_csv(powerbi_checklist_path, index=False)
print(powerbi_checklist_df.to_string(index=False))
print(f"\u2705 Saved -> {powerbi_checklist_path}")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: WORD REPORT -- POWERBI_DASHBOARD_REPORT.DOCX
# =============================================================================
_section("SECTION 13: Word Report -- PowerBI_Dashboard_Report.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = str(v)
    return table


report = Document()
report.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
report.add_paragraph("Power BI Dashboard Report -- Notebook 13")
report.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(report, "1. Scope & an Honest Limitation", level=1)
report.add_paragraph(
    "Power BI Desktop is a proprietary Windows GUI application with no Python library capable of producing a "
    "genuine, openable .pbix file -- this platform's standing rule is to never claim unsupported capability, "
    "so this notebook does not attempt to fabricate one. Instead it builds the real, import-ready data model "
    "(fact + dimension tables, CSV and Parquet), real DAX measures, a live-computed star schema diagram, and a "
    "static HTML preview of the intended report layout."
)

_add_heading(report, "2. Data Model -- Import Instructions", level=1)
for _step in [
    "In Power BI Desktop: Home -> Get Data -> Text/CSV (or Parquet) -> select fact_customer_risk_scores.csv/.parquet.",
    "Repeat Get Data for dim_risk_tier.csv, dim_model_version.csv, and dim_monitoring_window.csv.",
    "In Model view, draw the relationships listed in Section 3 below (dim tables on the 'one' side).",
    "In Model view -> New Measure, paste each measure from dax_measures.dax.",
    "Build visuals against the measures and dimension columns -- risk tier, model version, and monitoring window "
    "are all natural slicers.",
]:
    report.add_paragraph(_step, style="List Number")

_add_heading(report, "3. Star Schema Relationships (Live-Computed Cardinalities)", level=1)
report.add_picture(str(star_schema_chart_path), width=Inches(6.0))
_s_table = report.add_table(rows=1, cols=5)
_s_table.style = "Light Grid Accent 1"
_hdr = _s_table.rows[0].cells
_hdr[0].text, _hdr[1].text, _hdr[2].text, _hdr[3].text, _hdr[4].text = "From Table", "From Column", "To Table", "To Column", "Cardinality"
for _, r in star_schema_df.iterrows():
    c = _s_table.add_row().cells
    c[0].text, c[1].text, c[2].text, c[3].text, c[4].text = r["from_table"], r["from_column"], r["to_table"], r["to_column"], r["cardinality"]

_add_heading(report, "4. Risk Tier Distribution & Calibration (Real, Live-Computed)", level=1)
report.add_picture(str(chart1_path), width=Inches(6.0))
report.add_picture(str(chart2_path), width=Inches(6.0))
report.add_picture(str(chart3_path), width=Inches(6.0))

_add_heading(report, "5. DAX Measures", level=1)
report.add_paragraph(DAX_MEASURES)

_add_heading(report, "6. Power BI Readiness Checklist", level=1)
_c_table = report.add_table(rows=1, cols=3)
_c_table.style = "Light Grid Accent 1"
_hdr = _c_table.rows[0].cells
_hdr[0].text, _hdr[1].text, _hdr[2].text = "Dimension", "Status", "Evidence"
for r in powerbi_checklist:
    c = _c_table.add_row().cells
    c[0].text, c[1].text, c[2].text = r["dimension"], r["status"], r["evidence"]

report_path = POWERBI_DIR / "PowerBI_Dashboard_Report.docx"
report.save(str(report_path))
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 14: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Fact table row count matches the holdout population", len(fact_df) == X_holdout.shape[0],
       f"({len(fact_df)} vs {X_holdout.shape[0]})")
_check("Every fact row has a risk tier assigned", fact_df["risk_tier"].notna().all())
_check("Every fact row has a monitoring window id assigned", (fact_df["monitoring_window_id"] > 0).all())
_check("dim_risk_tier covers all 4 assumption bands", len(dim_risk_tier_df) == 4, f"({len(dim_risk_tier_df)})")
_check("Star schema has exactly 3 relationships (one per dimension)", len(star_schema_df) == 3, f"({len(star_schema_df)})")
_check("DAX measures file is non-empty and contains the expected measure count",
       N_DAX_MEASURES == len(DAX_MEASURE_DEFINITIONS) and N_DAX_MEASURES >= 8, f"({N_DAX_MEASURES} measures)")
_check("Parquet round-trips to the same row count as the CSV export",
       pl.read_parquet(str(fact_parquet_path)).shape[0] == len(fact_df))

_expected_files = [fact_csv_path, fact_parquet_path, dim_risk_tier_path, dim_model_version_path,
                    dim_monitoring_window_path, star_schema_chart_path, dax_measures_path,
                    chart1_path, chart2_path, chart3_path, html_preview_path, powerbi_checklist_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 13 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 13 checks passed.")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 15: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
    "fact_table_rows": int(len(fact_df)),
    "fact_csv_size_mb": round(fact_csv_path.stat().st_size / 1e6, 2),
    "fact_parquet_size_mb": round(fact_parquet_path.stat().st_size / 1e6, 2),
}
performance_report_path = ARTIFACTS_DIR / "notebook_13_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 15 complete.")


# =============================================================================
# SECTION 16: WRITE NOTEBOOK 13 SUMMARY ARTIFACT (for Notebook 17's rollup)
# =============================================================================
_section("SECTION 16: Write Notebook 13 Summary Artifact")

notebook_13_summary = {
    "notebook": "13_powerbi_dashboard", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "champion_model": CHAMPION_NAME, "fact_table_rows": int(len(fact_df)),
    "risk_tier_bands": PD_TIER_BANDS, "model_version_source": _model_version_source,
    "monitoring_window_source": _monitoring_window_source,
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb13_summary_path = ARTIFACTS_DIR / "notebook_13_summary.json"
with open(nb13_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_13_summary, f, indent=2)
print(f"\u2705 Saved -> {nb13_summary_path}")
print("\n\u2705 Section 16 complete.")


# =============================================================================
# SECTION 17: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 17: Notebook 13 Complete -- Handoff to Notebook 14")

print("NOTEBOOK 13: POWER BI DASHBOARD -- COMPLETE")
print(f"  Fact table rows                  : {len(fact_df):,}")
print(f"  Risk tiers (ASSUMPTION bands)     : {len(dim_risk_tier_df)}")
print(f"  Model version dimension           : {_model_version_source}")
print(f"  Monitoring window dimension       : {_monitoring_window_source}")
print(f"  DAX measures written              : {N_DAX_MEASURES}")
print(f"  Files produced                    : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb13_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                     : 14_executive_reports.ipynb")
print("\n\u2705 Ready to proceed.")
